In [ ]:
import os

print("Привет! Мы начинаем строить RAG-систему.")
print("Текущая рабочая директория:", os.getcwd())

# Функция для разбивки текста на куски (чанки)
def split_text_into_chunks(text, chunk_size=50, overlap=10):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Читаем наш тестовый файл (он лежит в папке data, а мы в notebooks, поэтому путь '../data/test.txt')
with open('../data/test.txt', 'r', encoding='utf-8') as f:
    document_text = f.read()

# Режем текст
chunks = split_text_into_chunks(document_text)

print(f"\nВсего символов в документе: {len(document_text)}")
print(f"Получилось кусков (чанков): {len(chunks)}\n")

# Выводим первые три куска, чтобы посмотреть, как это выглядит
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Чанк {i+1} ---")
    print(chunk)
    print()

In [ ]:
import re

# Функция для очистки текста от знаков препинания
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) # убираем знаки препинания
    return set(text.split()) # разбиваем на уникальные слова

# Функция поиска самого подходящего чанка
def find_best_chunk(query, chunks):
    query_words = clean_text(query)
    best_chunk = None
    max_matches = 0
    
    for chunk in chunks:
        chunk_words = clean_text(chunk)
        # Считаем, сколько слов из вопроса есть в этом куске текста
        matches = len(query_words.intersection(chunk_words))
        
        if matches > max_matches:
            max_matches = matches
            best_chunk = chunk
            
    return best_chunk, max_matches

# --- ПРОВЕРЯЕМ РАБОТУ ---
user_query = "Что такое RAG?"
print(f"Вопрос пользователя: {user_query}\n")

best_chunk, score = find_best_chunk(user_query, chunks)

print(f"Найден лучший чанк (совпадений: {score}):")
print(f"👉 {best_chunk}")

In [ ]:
def generate_answer(query, chunks):
    # 1. Находим нужный кусок текста (Retrieval)
    best_chunk, score = find_best_chunk(query, chunks)
    
    # 2. Формируем промпт (тот текст, который мы отправим нейросети)
    # Именно здесь мы объединяем найденный контекст и вопрос пользователя
    prompt = f"""
    Ты — полезный ассистент. Ответь на вопрос пользователя, используя ТОЛЬКО предоставленный контекст.
    Если в контексте нет ответа, скажи "Я не знаю".
    
    Контекст:
    {best_chunk}
    
    Вопрос: {query}
    
    Ответ:
    """
    
    return prompt

# --- ПРОВЕРЯЕМ ---
final_prompt = generate_answer("Что такое RAG?", chunks)

print("Вот что мы отправим нейросети (например, ChatGPT):")
print("-" * 50)
print(final_prompt)
print("-" * 50)

In [ ]:
import os
import requests
from dotenv import load_dotenv

# Загружаем ключи из .env
load_dotenv(override=True)

MY_API_KEY = os.getenv("YANDEX_API_KEY")
MY_FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")

if not MY_API_KEY or not MY_FOLDER_ID:
    print("❌ ОШИБКА: Ключи не найдены в файле .env")
else:
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    headers = {
        "Authorization": f"Api-Key {MY_API_KEY}",
        "Content-Type": "application/json",
        "x-folder-id": MY_FOLDER_ID
    }
    
    data = {
        "modelUri": f"gpt://{MY_FOLDER_ID}/yandexgpt-lite/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.3,
            "maxTokens": "500"
        },
        "messages": [
            {
                "role": "user",
                "text": final_prompt
            }
        ]
    }
    
    try:
        print("Отправляем запрос в YandexGPT...")
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        result = response.json()
        print("\n✅ ОТВЕТ НЕЙРОСЕТИ:")
        print("-" * 50)
        print(result['result']['alternatives'][0]['message']['text'])
        print("-" * 50)
    except Exception as e:
        print("❌ ОШИБКА:")
        print(e)